In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
metadata = {
    "nodes": {},
    "segments": {},
    "ways": {}
}

# Load data

In [3]:
PREPROCESS_ROOT = Path("../data/preprocess")

In [4]:
train_df = pd.read_csv("../data/raw/train.csv")

In [5]:
node_ids = sorted(
    set(train_df["s_node_id"]) |
    set(train_df["e_node_id"])
)
nodes_df = pd.read_csv(PREPROCESS_ROOT / "nodes.csv").sort_values("id")
nodes_df["id"] = nodes_df["id"].map(int)
nodes_df = nodes_df[nodes_df["id"].isin(node_ids)]
print(nodes_df.shape)

(11314, 13)


In [6]:
segment_ids = sorted(train_df["segment_id"])
segments_df = pd.read_csv(PREPROCESS_ROOT / "segments.csv").sort_values("id")
segments_df = segments_df[segments_df["id"].isin(segment_ids)]
segments_df["s_node_id"] = segments_df["s_node_id"].map(int)
segments_df["e_node_id"] = segments_df["e_node_id"].map(int)
print(segments_df.shape)


(10027, 36)


In [7]:
way_ids = sorted(train_df["street_id"])
ways_df = pd.read_csv(PREPROCESS_ROOT / "ways.csv").sort_values("id")
ways_df = ways_df[ways_df["id"].isin(way_ids)]
ways_df["id"] = ways_df["id"].map(int)
print(ways_df.shape)

(153, 15)


In [8]:
nodes_segments_edges_df = pd.read_csv(
    PREPROCESS_ROOT / "nodes_segments_edges_df.csv"
).sort_values("id")
nodes_segments_edges_df["id"] = nodes_segments_edges_df["id"].map(int)
nodes_segments_edges_df.shape

(10027, 3)

# Chuyển id -> index

In [9]:
node_id2index = dict()
node_index2id = dict()

for index, node_id in enumerate(nodes_df["id"]):
    node_id2index[node_id] = index
    node_index2id[index] = node_id

In [10]:
segment_id2index = dict()
segment_index2id = dict()

for index, segment_id in enumerate(segments_df["id"]):
    segment_id2index[segment_id] = index
    segment_index2id[index] = segment_id

In [11]:
way_id2index = dict()
way_index2id = dict()

for index, way_id in enumerate(ways_df["id"]):
    way_id2index[way_id] = index
    way_index2id[index] = way_id

In [12]:
nodes_df["id"] = nodes_df["id"].apply(
    lambda x: node_id2index[x]
)

In [13]:
segments_df["id"] = segments_df["id"].apply(
    lambda x: segment_id2index[x]
)
segments_df["s_node_id"] = segments_df["s_node_id"].apply(
    lambda x: node_id2index[x]
)
segments_df["e_node_id"] = segments_df["e_node_id"].apply(
    lambda x: node_id2index[x]
)

In [14]:
ways_df["id"] = ways_df["id"].apply(
    lambda x: way_id2index[x]
)

# Xây dựng đồ thị không đồng nhất

## Tĩnh

### Node

In [21]:
node_features = nodes_df.drop(columns=["id", "long", "lat"]).to_numpy()
nodes_df.columns

Index(['long', 'lat', 'id', 'tags.railway_level_crossing', 'tags.railway_no',
       'tags.railway_station', 'tags.junction_yes',
       'tags.crossing_traffic_signals', 'tags.crossing_zebra',
       'tags.highway_crossing', 'tags.highway_no',
       'tags.highway_traffic_signals', 'tags.bus_yes'],
      dtype='object')

### Segment

In [22]:
segments_df.columns

Index(['id', 'created_at', 'updated_at', 's_node_id', 'e_node_id', 'length',
       'street_id', 'max_velocity', 'street_level', 'name', 'type',
       'type_bus_station', 'type_car', 'type_cinema', 'type_clothes',
       'type_company', 'type_convenience', 'type_fuel', 'type_government',
       'type_house', 'type_marketplace', 'type_motorway', 'type_motorway_link',
       'type_pitch', 'type_primary', 'type_primary_link', 'type_residential',
       'type_school', 'type_secondary', 'type_secondary_link', 'type_tertiary',
       'type_tertiary_link', 'type_trunk', 'type_trunk_link',
       'type_unclassified', 'type_university'],
      dtype='object')